# Project 2: เดินผ่าน pipeline ทีละขั้น
เริ่มด้วย smoke data เพื่อตรวจระบบ ทุกขั้นเรียกโมดูลเดียวกับ CLI ไม่คัดลอก training logic ลง notebook
ติดตั้ง package ใน kernel นี้ก่อน: `python -m pip install -e ".[data,detection,dev]"` จาก project root

In [ ]:
from pathlib import Path
import os
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (root / "pyproject.toml").is_file(), "Open this notebook inside furniture-project2"
os.chdir(root)
from furniture.config import load_config, resolve_path
CONFIG = "configs/project_smoke.yaml"  # เปลี่ยนเป็น project.yaml เมื่อพร้อมข้อมูลจริง
EXPERIMENT = None  # ใช้ default ใน project config
cfg = load_config(CONFIG, EXPERIMENT)
{k: v for k, v in cfg.items() if k != "source_configs"}

## 1. เตรียม snapshot และ audit
ข้อมูลจริงต้องตรวจ source revision, labels และ groups ก่อนเทรน; ไม่เขียนทับ dataset version เดิม

In [ ]:
from furniture.smoke import make_smoke
from furniture.prepare import prepare_data, audit_data
data_root = resolve_path(cfg, cfg["dataset"]["root"])
if CONFIG.endswith("project_smoke.yaml") and not data_root.exists():
    make_smoke(data_root)
elif not data_root.exists():
    prepare_data(cfg)
audit = audit_data(cfg)
audit

## 2. ดู augmentation
ใช้เฉพาะ train; smoke/default อาจปิด augmentation ให้เปิดใน config หรือเลือก E3 เพื่อดูผล

In [ ]:
import uuid
from furniture.data import read_rows
from furniture.augmentation import preview
first = read_rows(data_root / "train.csv")[0]
preview_path = Path("artifacts/previews") / (uuid.uuid4().hex[:8] + ".png")
preview(data_root / first["image_path"], preview_path, cfg["model"]["image_size"], cfg["augmentation"])
from IPython.display import display, Image
display(Image(filename=str(preview_path)))

## 3. เทรนหนึ่งการทดลอง
สร้าง run ใหม่และเก็บ config/source/hash; ค่าคะแนน smoke ใช้ประเมินโค้ดเท่านั้น

In [ ]:
from furniture.pipeline import train_run
run = train_run(cfg)
run

## 4. ดู metrics แยกงาน
อ่าน validation ไม่ใช้ test ในการปรับโมเดล

In [ ]:
import json
run_dir = Path(run["run_dir"])
validation = json.loads((run_dir / "validation.json").read_text())
{task: {key: validation[task][key] for key in ("accuracy", "macro_f1")} for task in ("type", "style")}

## 5. ทดลองครบสามรูปแบบเมื่อเตรียมข้อมูลจริงแล้ว
CLI: `python -m furniture.cli experiments --config configs/project.yaml --ids E1 E2 E3`
E1 frozen control, E2 joint fine-tune, E3 + flip; ได้ comparison CSV อัตโนมัติ

## 6. Test หลังเลือก model และตรึง protocol
เปิดตัวแปรนี้เมื่อพร้อมประเมินครั้งสุดท้าย การเปรียบเทียบ test ต้องกำหนดวิธีไว้ก่อนและไม่ใช้ผลมา tune

In [ ]:
EVALUATE_TEST = False
if EVALUATE_TEST:
    from furniture.pipeline import evaluate_run
    test_results = evaluate_run(run_dir)
    display(test_results)
else:
    print("Test not opened. Review validation and freeze the protocol first.")

## 7. วิเคราะห์ภาพห้อง
ใช้ checkpoint ที่ฝึกเฟอร์นิเจอร์จริง ไม่ใช้ smoke weights
```bash
python -m furniture.cli room --config configs/project.yaml --image room.jpg --checkpoint RUN_DIR/best.pt --target scandinavian --catalog data/furniture-v1/train.csv --output artifacts/room-demo
```
คำแนะนำยังเป็น heuristic ต้อง human review และตรวจ product-to-room domain gap